# Mini Project 1 — Analysis Notebook

**Your name: Jared Ren**  
**Dataset: Youtube Livestream Chat Data from 4 League of Legends EMEA Champtionships (LEC) Livestreams**  
**Date: May 13, 2026**  

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [2]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import numpy as np
import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Dataset:** *(What is it? Where did it come from? Paste the URL or citation from your MP1a submission.)*

**Why this dataset:** *(One sentence connecting it to your HCD work or research interests.)*

**How this plan was adjusted (original vs collected data):**  
The first plan included a third question about **Super Chat** rates in high- vs low-volume chat periods (paid engagement vs “excitement”). After export, this LEC tournament chat has **only** `textMessageEvent` rows and **`super_chat_amount` is entirely empty**, so **paid messages are not present in the data** — not a coding mistake, but a **channel / broadcast setting** limitation. Rather than report a null comparison, **Question 3 was pivoted** to something the dataset can support: how **participation shape** (breadth of unique speakers vs share from the heaviest posters) differs between **high- and low-activity** time windows using the same 5-minute binning as Q1.

**Three analytical questions:**

1. How does chat message volume (messages per minute) change over the course of a single livestream, and at what timestamps do the highest-volume bursts occur? (Analyzed using time-interval binning and frequency counts in pandas.)
2. What share of total messages in a stream are contributed by the top 10% of most active users, and how concentrated is participation across a stream's chat audience? (Analyzed using value counts and cumulative distribution of messages per unique author ID.)
3. How does participation **breadth and dominance** differ between **high-** and **low-volume** 5-minute windows on the primary stream — e.g. average **unique authors** per bin and the **share of messages** in each bin that come from authors who rank in the stream’s **global top 10%** by message count? (Pivoted from Super Chat analysis; uses the same median split on bin volume as in the burst workflow.)

**What a practitioner would do with these findings:** *(One sentence. Who uses this, and for what?)*

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [3]:
# Load chat exports: prefer chat_regular/ next to this notebook, else cwd
from pathlib import Path

base_dir = Path.cwd()
chat_dir = base_dir / "chat_regular"
if any(chat_dir.glob("chat*.csv")):
    chat_files = sorted(chat_dir.glob("chat*.csv"))
else:
    chat_files = sorted(base_dir.glob("chat*.csv"))

if not chat_files:
    raise FileNotFoundError(
        f"No chat*.csv in {chat_dir} or {base_dir}. "
        "Add files to chat_regular/ or the notebook folder."
    )

if len(chat_files) == 1:
    df = pd.read_csv(chat_files[0])
    df["source_file"] = chat_files[0].name
else:
    df = pd.concat(
        [pd.read_csv(path).assign(source_file=path.name) for path in chat_files],
        ignore_index=True,
    )

df["stream_id"] = (
    df["source_file"].str.removesuffix(".csv").str.removeprefix("chat_")
)

# Longest broadcast by max chat offset (same rule as A5 chat_analysis.py)
PRIMARY_STREAM_ID = df.groupby("stream_id")["video_offset_sec"].max().idxmax()

print(f"Loaded {len(chat_files)} file(s):", [p.name for p in chat_files])
print("Primary stream for Q1–Q3:", PRIMARY_STREAM_ID)
print(df.shape)
df.head()

Loaded 4 file(s): ['chat__dE6Hddb8do.csv', 'chat_mPI1-aute0w.csv', 'chat_oxUSw1N9i3k.csv', 'chat_vUmNslJBba0.csv']
Primary stream for Q1–Q3: vUmNslJBba0
(20985, 11)


,timestamp,author_channel_id,display_name,message_text,message_sentiment,message_summary,message_type,super_chat_amount,video_offset_sec,source_file,stream_id
0,2026-05-02T13:46:52.641Z,UCRW-u9pyKpypEmjKKmvO_6A,@EMPtraxis,Looks like this is gonna be target practice fo...,neutral,looks like this,textMessageEvent,NaN,65.444,chat__dE6Hddb8do.csv,_dE6Hddb8do
1,2026-05-02T13:48:56.241Z,UCUaoe_b7_H7S1hN25er99eA,@georathor8814,if fnc goes 0-2 this week its byebye playoffs ...,questioning,question,textMessageEvent,NaN,188.985,chat__dE6Hddb8do.csv,_dE6Hddb8do
2,2026-05-02T13:49:34.666Z,UCYw246cyLEB3GprEEUBs2rw,@KILLERMIKE-1,wsp,neutral,wsp,textMessageEvent,NaN,227.565,chat__dE6Hddb8do.csv,_dE6Hddb8do
3,2026-05-02T13:50:26.812Z,UCTJYD_exfZg8UVjn80ROLqQ,@limeyman.,i like league saturdays,neutral,i like league,textMessageEvent,NaN,279.644,chat__dE6Hddb8do.csv,_dE6Hddb8do
4,2026-05-02T13:50:49.903Z,UCTJYD_exfZg8UVjn80ROLqQ,@limeyman.,lck into lpl into lec into lcs :D,neutral,lck into lpl,textMessageEvent,NaN,302.712,chat__dE6Hddb8do.csv,_dE6Hddb8do


In [4]:
# Check column types and missing values
df.info()

# Message types and Super Chat column (documents paid-chat absence; Q3 is pivoted)
print("\nmessage_type counts:")
print(df["message_type"].value_counts(dropna=False))
print("\nRows with non-null super_chat_amount:", df["super_chat_amount"].notna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 20985 entries, 0 to 20984
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   timestamp          20985 non-null  str    
 1   author_channel_id  20985 non-null  str    
 2   display_name       20985 non-null  str    
 3   message_text       20842 non-null  str    
 4   message_sentiment  20985 non-null  str    
 5   message_summary    20985 non-null  str    
 6   message_type       20985 non-null  str    
 7   super_chat_amount  0 non-null      float64
 8   video_offset_sec   20985 non-null  float64
 9   source_file        20985 non-null  str    
 10  stream_id          20985 non-null  str    
dtypes: float64(2), str(9)
memory usage: 1.8 MB

message_type counts:
message_type
textMessageEvent    20985
Name: count, dtype: int64

Rows with non-null super_chat_amount: 0


In [ ]:
# Summary statistics for numeric columns
df.describe()

,super_chat_amount,video_offset_sec
count,0.0,20985.000000
mean,NaN,13783.410140
std,NaN,6325.694667
min,NaN,48.255000
25%,NaN,8221.753000
50%,NaN,13555.833000
75%,NaN,18621.880000
max,NaN,28022.668000


**Your data profile notes:**  

Each row is one live chat message. Key columns: **`timestamp`** (UTC), **`video_offset_sec`** (seconds from stream start for binning and Q1), **`author_channel_id`** (stable ID for Q2 and Q3), **`message_text`**, **`message_type`**, **`super_chat_amount`**. **`stream_id`** / **`source_file`** identify which VOD the row came from.

Focus for this notebook: **time** and **author** fields for volume (Q1), inequality (Q2), and **who speaks during busy vs quiet windows** (Q3). Check **`message_type`** / **`super_chat_amount`** in the profile: if there are no Super Chats, Section 1 explains the **pivot** for Q3 away from paid engagement.

- **Shape:** 20,985 combined rows and 11 columns after concatenating the four `chat_regular` CSVs and adding `source_file` / `stream_id`.
- **Per stream (`stream_id`):** `_dE6Hddb8do` 3,438 rows; `mPI1-aute0w` 3,625; `oxUSw1N9i3k` 5,951; `vUmNslJBba0` 7,971.
- **`author_channel_id`:** 0 nulls in this export.
- **`super_chat_amount`:** no populated numeric values (all missing or empty in this pull). **`message_type`:** only `textMessageEvent` (20,985 rows), with no `superChatEvent` rows here.
- **Other checks:** `video_offset_sec` is present for binning; timestamps are ISO-style UTC strings in the files used here. Optional text fields (for example sentiment labels) are filled but are outside the questions in this notebook.

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** How does chat message volume (messages per minute) change over the course of a single livestream, and at what timestamps do the highest-volume bursts occur? (Analyzed using time-interval binning and frequency counts in pandas.)

In [5]:
# Q1 — Volume (messages/min) and burst timestamps on the primary stream
# 5-minute bins: messages_per_min = count_in_bin / 5

primary = df[df["stream_id"] == PRIMARY_STREAM_ID].copy()
primary["offset_min_bin_5"] = (primary["video_offset_sec"] // 300) * 5
primary["timestamp"] = pd.to_datetime(
    primary["timestamp"], utc=True, errors="coerce"
)
_bad_ts = int(primary["timestamp"].isna().sum())
if _bad_ts:
    print(
        f"Warning: {_bad_ts} row(s) could not parse as timestamps (coerced to NaT)."
    )

bin_counts = (
    primary.groupby("offset_min_bin_5", as_index=False)
    .size()
    .rename(columns={"size": "messages_in_bin"})
)
bin_counts["messages_per_min"] = bin_counts["messages_in_bin"] / 5.0

# Wall-clock timestamps for each bin (first and last message in the window)
ts_bounds = primary.groupby("offset_min_bin_5")["timestamp"].agg(
    first_ts="min", last_ts="max"
).reset_index()

q1_summary = bin_counts.merge(ts_bounds, on="offset_min_bin_5", how="left")

median_mpm = bin_counts["messages_per_min"].median()
print(f"Primary stream: {PRIMARY_STREAM_ID}  |  messages: {len(primary):,}")
print(f"Median messages/min (5-min bins): {median_mpm:.1f}")
print("\nTop 5 burst windows (by messages/min):")
top5 = q1_summary.nlargest(5, "messages_per_min")[
    [
        "offset_min_bin_5",
        "messages_in_bin",
        "messages_per_min",
        "first_ts",
        "last_ts",
    ]
]
print(top5.to_string(index=False))


Primary stream: vUmNslJBba0  |  messages: 7,971
Median messages/min (5-min bins): 15.8

Top 5 burst windows (by messages/min):
 offset_min_bin_5  messages_in_bin  messages_per_min                 first_ts                  last_ts
            385.0              239              47.8 2026-04-26T20:11:10.939Z 2026-04-26T20:16:10.718Z
            320.0              209              41.8 2026-04-26T19:06:12.394Z 2026-04-26T19:11:04.683Z
            370.0              198              39.6 2026-04-26T19:56:14.165Z 2026-04-26T20:01:10.564Z
            445.0              190              38.0 2026-04-26T21:11:11.054Z 2026-04-26T21:16:08.956Z
            250.0              176              35.2 2026-04-26T17:56:11.552Z 2026-04-26T18:01:08.728Z


**Interpretation:**  
Message rate on the primary VOD is uneven relative to the overall bin median of 15.8 messages per minute. The strongest 5-minute window begins at offset minute 385 and reaches 47.8 messages per minute with 239 messages in that bin, while the next four peaks in the printed top five start near minutes 320, 370, 445, and 250 at about 41.8 down to 35.2 messages per minute. That pattern is closer to short bursts than a flat background rate across the timeline. The paired `first_ts` and `last_ts` values only locate those bursts in UTC wall time and do not support claims about what happened in the game.


**Question 2:** What share of total messages in a stream are contributed by the top 10% of most active users, and how concentrated is participation across a stream's chat audience? (Analyzed using value counts and cumulative distribution of messages per unique author ID.)

In [6]:
# Q2 — Top 10% of authors by activity and cumulative concentration (primary stream)

primary_msgs = df[df["stream_id"] == PRIMARY_STREAM_ID]
_dropped_author = int(primary_msgs["author_channel_id"].isna().sum())
primary_msgs = primary_msgs.dropna(subset=["author_channel_id"])
if _dropped_author:
    print(
        f"Warning: dropped {_dropped_author} row(s) with missing author_channel_id "
        "before author-based counts."
    )
counts = primary_msgs["author_channel_id"].value_counts()
n_users = counts.size
total_msgs = counts.sum()

k_top10 = max(1, int(np.ceil(0.10 * n_users)))
msgs_from_top10pct = counts.head(k_top10).sum()
share_top10 = msgs_from_top10pct / total_msgs

print(f"Primary stream: {PRIMARY_STREAM_ID}")
print(f"Unique authors: {n_users:,}  |  Total messages: {total_msgs:,}")
print(f"Top 10% of users ≈ {k_top10} users (by message rank)")
print(f"Share of all messages from those users: {100 * share_top10:.1f}%")

# Cumulative share of messages as you include more users (most active first)
ranked = counts.sort_values(ascending=False).reset_index()
ranked.columns = ["author_channel_id", "messages"]
ranked["cum_msgs"] = ranked["messages"].cumsum()
ranked["cum_share"] = ranked["cum_msgs"] / total_msgs
ranked["user_rank_pct"] = (np.arange(1, len(ranked) + 1) / n_users) * 100

def cum_share_at_user_pct(pct):
    """Fraction of all messages from the most active ceil(pct * n_users) users."""
    n_take = max(1, int(np.ceil((pct / 100) * n_users)))
    return ranked.loc[n_take - 1, "cum_share"]

for p in (10, 25, 50):
    print(
        f"Cumulative message share from top {p}% of users (by count): "
        f"{100 * cum_share_at_user_pct(p):.1f}%"
    )

ranked.head(10)


Primary stream: vUmNslJBba0
Unique authors: 1,187  |  Total messages: 7,971
Top 10% of users ≈ 119 users (by message rank)
Share of all messages from those users: 65.4%
Cumulative message share from top 10% of users (by count): 65.4%
Cumulative message share from top 25% of users (by count): 81.7%
Cumulative message share from top 50% of users (by count): 91.9%


,author_channel_id,messages,cum_msgs,cum_share,user_rank_pct
0,UCULsdOaoafN2q6OB2gJWvUA,307,307,0.038515,0.084246
1,UCSvjQBDgYDB5TGVmCZObcwA,218,525,0.065864,0.168492
2,UC2TViDabpTmxwYCNbOntgqw,191,716,0.089826,0.252738
3,UChDbdTtfaDvJE0-8GRNCTpQ,170,886,0.111153,0.336984
4,UCaUrKWNHTVbzVb1VqVBTYRg,129,1015,0.127337,0.421230
5,UCXqOfa7BCzWjaVxYBUuGlMQ,119,1134,0.142266,0.505476
6,UCZC1b1s4EvtmPqxeZC_Ovfg,102,1236,0.155062,0.589722
7,UCSjppbG42UVcJbI-O5SCxBw,100,1336,0.167608,0.673968
8,UCy8nCuu1OrW6cHFI1rfHXRw,94,1430,0.179400,0.758214
9,UCsY7bUrHvFfjFLCE4Ha0afw,87,1517,0.190315,0.842460


**Interpretation:**  
On the primary stream the table summarizes 7,971 messages from 1,187 distinct authors, and the busiest tenth of accounts by line count (119 users after the ceiling rule) account for about 65.4% of all messages. The cumulative splits tighten quickly: about 81.7% of lines come from the busiest quarter of accounts, and about 91.9% from the busiest half, which reads as a strong heavy tail in posting volume. That describes inequality in how often pseudonymous IDs speak, not motives or identities behind those keys.


**Question 3:** How does participation breadth and dominance differ between high- and low-volume 5-minute windows on the primary stream (unique authors per bin and share of messages from globally top-10% posters)? *(Pivoted from Super Chat vs activity because this export has no paid messages.)*

In [7]:
# Q3 — Participation in high vs low activity bins (pivot: no Super Chats in this export)
# Same 5-minute bins and median split on bin message count as in Q1.
# Metrics per bin: unique authors; share of messages from stream-wide top 10% posters.

p3 = df[df["stream_id"] == PRIMARY_STREAM_ID].copy()
p3["offset_min_bin_5"] = (p3["video_offset_sec"] // 300) * 5

author_totals = p3.groupby("author_channel_id").size().sort_values(ascending=False)
n_authors = author_totals.size
k_top = max(1, int(np.ceil(0.10 * n_authors)))
top10_authors = set(author_totals.head(k_top).index)
p3["from_global_top10pct"] = p3["author_channel_id"].isin(top10_authors)

per_bin = (
    p3.groupby("offset_min_bin_5", as_index=False)
    .agg(
        messages_in_bin=("author_channel_id", "size"),
        unique_authors=("author_channel_id", pd.Series.nunique),
        share_msgs_from_top10=("from_global_top10pct", "mean"),
    )
)
median_count = per_bin["messages_in_bin"].median()
per_bin["activity_level"] = np.where(
    per_bin["messages_in_bin"] >= median_count, "high", "low"
)

q3_summary = (
    per_bin.groupby("activity_level", observed=True)
    .agg(
        n_bins=("offset_min_bin_5", "count"),
        avg_messages_per_bin=("messages_in_bin", "mean"),
        avg_unique_authors_per_bin=("unique_authors", "mean"),
        avg_share_msgs_from_global_top10=("share_msgs_from_top10", "mean"),
    )
)

print(f"Primary stream: {PRIMARY_STREAM_ID}")
print(f"Median messages per 5-min bin (split threshold): {median_count:.1f}")
print(
    "\nHigh vs low activity bins — breadth (unique authors) and "
    "dominance (share of lines from global top-10% posters):"
)
print(q3_summary.to_string())
print(
    "\n(Original plan: Super Chat rate by activity; not applicable — "
    "no superChatEvent / no super_chat_amount in this dataset.)"
)


Primary stream: vUmNslJBba0
Median messages per 5-min bin (split threshold): 79.0

High vs low activity bins — breadth (unique authors) and dominance (share of lines from global top-10% posters):
                n_bins  avg_messages_per_bin  avg_unique_authors_per_bin  avg_share_msgs_from_global_top10
activity_level                                                                                            
high                48            123.875000                   64.479167                           0.65357
low                 46             44.021739                   22.608696                           0.68174

(Original plan: Super Chat rate by activity; not applicable — no superChatEvent / no super_chat_amount in this dataset.)


**Interpretation:**  
Bins are split at the median message count per 5-minute window, 79 messages, yielding 48 high-activity bins and 46 low bins on the primary stream. High bins average about 123.9 messages and about 64.5 unique authors per bin, compared with about 44.0 messages and 22.6 unique authors in low bins, so louder periods clearly involve more distinct speakers on average. The mean share of lines from the globally defined busiest tenth of accounts is about **65.4%** in high bins versus about **68.2%** in low bins, so dominance by those heavy posters is high in both groups but slightly lower when bins are busier. In practical terms, busier windows look like broader participation rather than heavy posters alone driving the spike, though those heavy accounts still contribute the majority of lines in both regimes.


---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

### Story these charts tell

On this **official LEC** primary stream, chat is **not steady**: a few **short windows** carry far more messages than the rest. Among people, a **small top slice** of accounts writes a **large share** of everything. When windows are **busier**, you usually see **more different people** speaking—but **heavy posters** can still account for a big fraction of lines. Together that is: **spiky in time**, **unequal across people**, and **busy moments are still “crowded” in a narrow sense**—useful for producers thinking about where attention clusters (and why co-streams can feel louder elsewhere).

In [13]:
# Section 4 — Three simple charts, one story (see markdown above)
# Needs: load cell ran (df, PRIMARY_STREAM_ID, numpy as np, pandas as pd)

import plotly.express as px
primary = df[df["stream_id"] == PRIMARY_STREAM_ID].copy()
primary["offset_min_bin_5"] = (primary["video_offset_sec"] // 300) * 5
_dropped_sec4 = int(primary["author_channel_id"].isna().sum())
primary_for_who = primary.dropna(subset=["author_channel_id"])
if _dropped_sec4:
    print(
        f"Warning: dropped {_dropped_sec4} row(s) with missing author_channel_id "
        "before author-based charts."
    )

bin_df = (
    primary.groupby("offset_min_bin_5", as_index=False)
    .size()
    .rename(columns={"size": "messages_in_bin"})
)
bin_df["messages_per_min"] = bin_df["messages_in_bin"] / 5.0

# --- Chart 1: WHEN did chat spike? (easy: only the top windows, horizontal bars) ---
top_n = 12
top_bins = bin_df.nlargest(top_n, "messages_per_min").sort_values("messages_per_min")
top_bins["time_slice"] = top_bins["offset_min_bin_5"].apply(
    lambda m: f"Minute {int(m)} → {int(m + 5)}"
)

fig1 = px.bar(
    top_bins,
    x="messages_per_min",
    y="time_slice",
    orientation="h",
    title="When was chat fastest? (largest 5-minute slices only)",
    labels={
        "messages_per_min": "Messages each minute (average inside that slice)",
        "time_slice": "",
    },
)
fig1.update_layout(
    yaxis={"categoryorder": "total ascending"},
    annotations=[
        dict(
            text="Horizontal axis = messages per minute (count in each 5-min window ÷ 5).",
            xref="paper",
            yref="paper",
            x=0,
            y=-0.16,
            showarrow=False,
            xanchor="left",
            font=dict(size=11),
        )
    ],
    margin=dict(b=70),
)
fig1.show()

# --- Chart 2: WHO writes most lines? (two bars: top 10% vs everyone else) ---
counts = primary_for_who["author_channel_id"].value_counts()
n_users = counts.size
total = counts.sum()
k_top = max(1, int(np.ceil(0.10 * n_users)))
pct_top = 100 * counts.head(k_top).sum() / total
who_df = pd.DataFrame(
    {
        "Group": ["Top 10% busiest people", "All other people combined"],
        "Percent of all messages": [pct_top, 100 - pct_top],
    }
)

fig2 = px.bar(
    who_df,
    x="Group",
    y="Percent of all messages",
    title="Who writes most of the chat on this stream?",
    color="Group",
    color_discrete_map={
        "Top 10% busiest people": "#EF553B",
        "All other people combined": "#636efa",
    },
    text="Percent of all messages",
)
fig2.update_traces(texttemplate="%{y:.1f}%", textposition="outside")
fig2.update_yaxes(
    title="Percent of all messages on this stream.",
    range=[0, max(100, pct_top * 1.15)],
)
fig2.update_layout(showlegend=False)
fig2.show()

# --- Chart 3: In CALM vs BUSY slices, crowd size vs heavy posters (Question 3) ---
from math import ceil

p3 = primary.copy()
author_totals = p3.groupby("author_channel_id").size().sort_values(ascending=False)
k10 = max(1, ceil(0.10 * author_totals.size))
heavy = set(author_totals.head(k10).index)
p3["from_heavy_posters"] = p3["author_channel_id"].isin(heavy)

per_bin = (
    p3.groupby("offset_min_bin_5", as_index=False)
    .agg(
        messages_in_bin=("author_channel_id", "size"),
        different_people=("author_channel_id", pd.Series.nunique),
        pct_lines_from_heavy=("from_heavy_posters", "mean"),
    )
)
mid = per_bin["messages_in_bin"].median()
per_bin["kind"] = np.where(
    per_bin["messages_in_bin"] >= mid,
    "Busier half of the stream (by slice)",
    "Quieter half of the stream (by slice)",
)

summary = (
    per_bin.groupby("kind", observed=True)
    .agg(
        avg_different_people=("different_people", "mean"),
        avg_pct_from_heavy=("pct_lines_from_heavy", "mean"),
    )
    .reset_index()
)

fig3a = px.bar(
    summary,
    x="kind",
    y="avg_different_people",
    title="Q3a — In calmer vs busier slices: how many different people chat?",
    labels={
        "kind": "",
        "avg_different_people": "Average number of different people (per 5-minute slice)",
    },
)
fig3a.update_layout(
    annotations=[
        dict(
            text=(
                "Quieter/busier = bins below/above the median message count per window "
                "(not first vs second half of runtime)."
            ),
            xref="paper",
            yref="paper",
            x=0,
            y=-0.2,
            showarrow=False,
            xanchor="left",
            font=dict(size=10),
        )
    ],
    margin=dict(b=85),
)
fig3a.show()

summary["pct_lines_heavy"] = 100 * summary["avg_pct_from_heavy"]

fig3b = px.bar(
    summary,
    x="kind",
    y="pct_lines_heavy",
    title="Q3b — What share of lines come from the stream's heaviest 10% of posters?",
    labels={
        "kind": "",
        "pct_lines_heavy": "Average percent of lines in that slice",
    },
)
fig3b.update_yaxes(title="Percent of lines in that slice", tickformat=".0f")
fig3b.update_layout(
    annotations=[
        dict(
            text=(
                "Quieter/busier = bins below/above the median message count per window "
                "(not first vs second half of runtime)."
            ),
            xref="paper",
            yref="paper",
            x=0,
            y=-0.2,
            showarrow=False,
            xanchor="left",
            font=dict(size=10),
        )
    ],
    margin=dict(b=85),
)
fig3b.show()


**Chart rationale:**

**Chart 1 — When**  
This is only the **twelve loudest** 5-minute slices, as a horizontal bar chart. Longer bar = faster chat in that slice. It answers “when should a human look?” without reading a wiggly line.

**Chart 2 — Who**  
Two bars split **all messages** into (a) the busiest **10% of accounts** on this stream and (b) **everyone else**. If the red bar is tall, a **small club** is doing most of the typing.

**Chart 3 — Calm vs busy slices**  
Each 5-minute slice is labeled **quieter** or **busier** than the median slice. **First small chart:** average **head count** (different people) in quieter vs busier slices. **Second small chart:** average **percent of lines** in that slice typed by the stream’s **heaviest 10%** of posters. Two charts so each axis has one clear meaning (we do **not** mix counts and percents on one scale). Same median split as Section 3.


**Static chart exports:** In the `Week 6/` folder (next to this notebook), `export_mp1_charts.py` writes six files for the same three MP1a questions as Section 3: `mp1_q1_volume_bursts.png` / `.svg`, `mp1_q2_author_concentration.png` / `.svg`, and `mp1_q3_participation_high_vs_low.png` / `.svg`. Each figure title states what is shown, and axis labels include units (percent, messages per minute, author counts, or minute-based window labels). Re-run the script from that folder after replacing the CSV inputs.

**Uninteresting result (not plotted):** The pre-pivot Super Chat question would not yield an informative chart here because this pull has only `textMessageEvent` rows and no usable `super_chat_amount`, so a paid-vs-activity visualization would be empty by construction. The exported Question 3 figure uses the participation pivot documented in Section 1.


---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**  

**Pivot summary (why and how):**  
I originally planned to compare **Super Chat** frequency in high- vs low-volume chat periods. The collected LEC chat exports contain **no** `superChatEvent` rows and **no** usable `super_chat_amount` values, so **paid engagement cannot be measured** on this channel. Instead of forcing a null result as if it were a finding, I **replaced Question 3** with an analysis of **free-chat participation shape**: in each 5-minute bin (same binning as Q1), how many **unique authors** speak and what **share of lines** come from the stream’s globally busiest 10% of accounts, compared between **high-** and **low-activity** bins (median split). That keeps the “excitement window” framing while matching what the data actually records.

**Reflection — denser chat and where the audience sits:**  
These official tournament VODs felt **relatively light** in chat density compared with what I imagined for peak esports hype. Next time I will try to collect **richer, denser** chat streams—e.g. **popular co-streams** where creators watch or react to the same matches—because much of the **live energy** (higher message rates, more simultaneous chatters) often pools there rather than on the **official** channel alone. That is a sampling choice, not a verdict on LEC itself: it reflects **where viewers choose to hang out**, and matching analysis goals to those hubs would give stronger material for volume- and participation-style questions.

**Draft conclusion (for review before final submission):**

On the primary stream, message rates in 5-minute windows are uneven: the busiest slices approach about 48 messages per minute (47.8) while the median is roughly 16 (15.8), so bursts are short-lived rather than flat background chatter. Author activity is similarly skewed: the busiest tenth of accounts account for about 65% of all chat lines on this VOD. After the Super Chat pivot, the table shows that above-median bins involve far more unique authors than quieter bins, while the share of lines from the stream-wide heaviest tenth of posters remains high in both groups and is modestly lower in the busiest bins, which reads as larger crowds mixing in rather than heavy posters alone carrying every loud window. What stood out relative to that pattern is that quieter bins register a slightly larger line share from those heavy accounts despite fewer distinct speakers, which sharpens how both volume and participation should be read together. Going forward I would probe denser venues such as co-stream chat for comparison, and I would keep claims descriptive: these logs cannot support Super Chat effects, cannot be tied to specific in-game moments, and describe behavior on this official export only.

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.